# Layer RAW — Ingesta de CSVs a Parquet

**Objetivo:** Leer los CSVs originales de Ecobici BA (recorridos y usuarios 2022-2024) y guardarlos en formato Parquet sin transformaciones de negocio.

**Pipeline Medallion:**
```
CSV (Datasets/) ──► RAW (Parquet) ──► STG ──► MART_ML / MART_BI
```

**Estructura esperada del repo:**
```
proyecto/
├── Datasets/      ← CSVs originales
├── data/          ← generado al correr los notebooks
└── notebooks/     ← estos notebooks
```

In [ ]:
import pandas as pd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Path().resolve() en Jupyter apunta al directorio del notebook (notebooks/).
# .parent sube al root del proyecto. Funciona en cualquier maquina sin cambiar nada.
BASE_DIR     = Path().resolve().parent
DATASETS_DIR = BASE_DIR / "Datasets"
RAW_DIR      = BASE_DIR / "data" / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Directorios:")
print(f"  Fuente  : {DATASETS_DIR}")
print(f"  RAW out : {RAW_DIR}")

In [ ]:
def csv_to_parquet(csv_path: Path, parquet_path: Path,
                   encoding: str = 'utf-8', chunksize: int = 400_000,
                   **read_kwargs) -> tuple:
    """Lee un CSV grande en chunks y lo guarda como Parquet.
    
    Todos los campos se leen como str (dtype=str) para preservar
    el dato crudo; la tipificacion ocurre en la capa STG.
    """
    size_mb = csv_path.stat().st_size / 1024**2
    print(f"\n  Archivo : {csv_path.name}  ({size_mb:.0f} MB)")

    chunks = []
    for i, chunk in enumerate(
        pd.read_csv(csv_path, encoding=encoding, chunksize=chunksize,
                    dtype=str, low_memory=False, **read_kwargs)
    ):
        chunks.append(chunk)
        print(f"    chunk {i+1}: {len(chunk):,} filas", end='\r')

    df = pd.concat(chunks, ignore_index=True)
    print(f"    Total  : {df.shape[0]:,} filas x {df.shape[1]} cols      ")

    df.to_parquet(parquet_path, index=False, compression='snappy')
    out_mb = parquet_path.stat().st_size / 1024**2
    print(f"    Parquet: {parquet_path.name}  ({out_mb:.0f} MB)")
    return df.shape

## Recorridos

In [ ]:
print("=== RECORRIDOS 2024 ===")
csv_to_parquet(
    csv_path     = DATASETS_DIR / "badata_ecobici_recorridos_realizados_2024.csv",
    parquet_path = RAW_DIR / "recorridos_2024.parquet"
)

In [ ]:
print("=== RECORRIDOS 2022 ===")
csv_to_parquet(
    csv_path     = DATASETS_DIR / "trips_2022.csv",
    parquet_path = RAW_DIR / "recorridos_2022.parquet",
    encoding     = 'utf-8'
)

In [ ]:
print("=== RECORRIDOS 2023 ===")
csv_to_parquet(
    csv_path     = DATASETS_DIR / "trips_2023.csv",
    parquet_path = RAW_DIR / "recorridos_2023.parquet",
    encoding     = 'utf-8'
)

## Usuarios

In [ ]:
print("=== USUARIOS 2022 ===")
csv_to_parquet(
    csv_path     = DATASETS_DIR / "usuarios_ecobici_2022.csv",
    parquet_path = RAW_DIR / "usuarios_2022.parquet"
)

print("=== USUARIOS 2023 ===")
csv_to_parquet(
    csv_path     = DATASETS_DIR / "usuarios_ecobici_2023.csv",
    parquet_path = RAW_DIR / "usuarios_2023.parquet"
)

print("=== USUARIOS 2024 ===")
csv_to_parquet(
    csv_path     = DATASETS_DIR / "usuarios_ecobici_2024.csv",
    parquet_path = RAW_DIR / "usuarios_2024.parquet"
)

## Verificación final

In [ ]:
print("=" * 60)
print("RESUMEN — RAW LAYER")
print("=" * 60)
total_mb = 0
for f in sorted(RAW_DIR.glob("*.parquet")):
    mb = f.stat().st_size / 1024**2
    total_mb += mb
    df_peek = pd.read_parquet(f)
    print(f"\n{f.name}  ({mb:.0f} MB)")
    print(f"  Shape  : {df_peek.shape}")
    print(f"  Cols   : {list(df_peek.columns)}")

print(f"\nTotal en disco: {total_mb:.0f} MB")
print("\nRAW layer completo.")